## Обучаем ResNet на ббоксах

## Подключаю колаб

In [1]:
!pip3 install pytorch_lightning albumentations torchvision timm rapidfuzz boto3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.3/857.3 kB 61.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 126.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.6/140.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.8/14.8 MB 83.8 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.8/86.8 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 67.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
local_folder = '/content/drive/MyDrive/sneakers-hse/03_yolo_preprocessed'
!git clone https://github.com/xan-d-or/sneakers-hse.git
%cd ./sneakers-hse/
!pwd
!git checkout yolo

fatal: destination path 'sneakers-hse' already exists and is not an empty directory.
/content/sneakers-hse
/content/sneakers-hse
Branch 'yolo' set up to track remote branch 'yolo' from 'origin'.
Switched to a new branch 'yolo'


In [5]:
import sys
sys.path.append('../..')

from pathlib import Path
import os
from glob import glob

import pandas as pd
import numpy as np

from PIL import Image

from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import torch
import pytorch_lightning as pl
from torch.utils.data import DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

from src.model.baseline_cnn import LitCNN
from src.model.resnet_18 import LitResNet18
from src.model.dataset import ImageDataset
from src.model.classifier import LitClassifier

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping

import fsspec

from tqdm import tqdm
tqdm.pandas()


In [6]:
!ls /content/drive/MyDrive/sneakers-hse/03_yolo_preprocessed/

search-dataset-images  test_dataset  test_images.csv  train_images.csv


In [7]:
train_df_pre = pd.read_csv(local_folder + '/train_images.csv')
display(train_df_pre.head(), train_df_pre.shape)

test_df = pd.read_csv(local_folder + '/test_images.csv')
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,0,Vans Кеды Upland/clothing_0_264.jpeg,Vans Кеды Upland,0
1,1,Vans Кеды Upland/clothing_0_57.jpeg,Vans Кеды Upland,0
2,2,Vans Кеды Upland/orig_45.jpeg,Vans Кеды Upland,0
3,3,Vans Кеды Upland/clothing_0_0.jpeg,Vans Кеды Upland,0
4,4,Vans Кеды Upland/clothing_0_233.jpeg,Vans Кеды Upland,0


(10965, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [8]:
train_df, val_df = train_test_split(
    train_df_pre,
    test_size=0.2,
    stratify=train_df_pre["sneaker_class"],
    random_state=42
)

display(train_df.head(), train_df.shape)
display(val_df.head(), val_df.shape)
display(test_df.head(), test_df.shape)

,Unnamed: 0,path,sneaker_class,corrupted_flg
2195,2288,Nike Кеды Dunk Low Retro/clothing_0_103.jpeg,Nike Кеды Dunk Low Retro,0
10557,10855,PUMA Кроссовки Puma Morphic/orig_111.jpeg,PUMA Кроссовки Puma Morphic,0
7299,7477,Kari Кроссовки/clothing_0_190.jpeg,Kari Кроссовки,0
4103,4230,Reebok Кроссовки CLASSIC LEATHER/clothing_0_43...,Reebok Кроссовки CLASSIC LEATHER,0
2097,2188,Nike Кеды Dunk Low Retro/clothing_0_86.jpeg,Nike Кеды Dunk Low Retro,0


(8772, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
7461,7639,Vans Кеды Knu Skool/clothing_0_98.jpeg,Vans Кеды Knu Skool,0
2486,2591,Reebok Кроссовки CLASSIC NYLON/orig_291.jpeg,Reebok Кроссовки CLASSIC NYLON,0
4889,5027,Nike Кроссовки AIR MAX 90/clothing_0_12.jpeg,Nike Кроссовки AIR MAX 90,0
2655,2762,Under Armour Кроссовки UA CHARGED SPEED SWIFT/...,Under Armour Кроссовки UA CHARGED SPEED SWIFT,0
231,240,Vans Кеды Upland/clothing_0_62.jpeg,Vans Кеды Upland,0


(2193, 4)

,Unnamed: 0,path,sneaker_class,corrupted_flg
0,14,Vans Кеды Upland/clothing_0_168_real.jpeg,Vans Кеды Upland,0
1,32,Vans Кеды Upland/clothing_0_215_real.jpeg,Vans Кеды Upland,0
2,44,Vans Кеды Upland/orig_216_real.jpeg,Vans Кеды Upland,0
3,80,Vans Кеды Upland/shoe_3_100_real.jpeg,Vans Кеды Upland,0
4,87,Vans Кеды Upland/clothing_0_277_real.jpeg,Vans Кеды Upland,0


(300, 4)

In [9]:
train_paths = train_df["path"].tolist()
val_paths   = val_df["path"].tolist()
test_paths  = test_df["path"].tolist()

train_labels = train_df["sneaker_class"].tolist()
val_labels   = val_df["sneaker_class"].tolist()
test_labels  = test_df["sneaker_class"].tolist()

In [10]:
# Аугментация и приведение всех изображений к одному масштабу

train_tfms = A.Compose([
    A.Resize(224, 224),
    A.HorizontalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

val_tfms = A.Compose([
    A.Resize(224, 224),
    A.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225)
    ),
    ToTensorV2(),
])

In [11]:
path_to_dataset = local_folder + '/search-dataset-images'

In [12]:
train_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=train_paths,
    labels=train_labels,
    augmenter=train_tfms
)

val_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=val_paths,
    labels=val_labels,
    augmenter=val_tfms
)

test_dataset = ImageDataset(
    base_path=path_to_dataset,
    images_path=test_paths,
    labels=test_labels,
    augmenter=val_tfms
)

In [17]:
num_workers = 2
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=False,
    persistent_workers=True,
)

In [18]:
callbacks = [
    ModelCheckpoint(
        monitor="val_loss",
        save_top_k=1,
        mode="min"
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=3
    )
]

In [19]:
from src.data.utils.eda_utils import directory_to_dataframe

df = directory_to_dataframe(path_to_dataset)
df

,path,sneaker_class
0,Kari Кеды/clothing_0_214.jpeg,Kari Кеды
1,Kari Кеды/clothing_0_10.jpeg,Kari Кеды
2,Kari Кеды/clothing_0_147.jpeg,Kari Кеды
3,Kari Кеды/clothing_0_165.jpeg,Kari Кеды
4,Kari Кеды/clothing_0_106.jpeg,Kari Кеды
...,...,...
11260,adidas Кроссовки adizero Evo SL M/shoe_3_12.jpeg,adidas Кроссовки adizero Evo SL M
11261,adidas Кроссовки adizero Evo SL M/shoe_3_120_r...,adidas Кроссовки adizero Evo SL M
11262,adidas Кроссовки adizero Evo SL M/shoe_1_22_re...,adidas Кроссовки adizero Evo SL M
11263,adidas Кроссовки adizero Evo SL M/shoe_7_120_r...,adidas Кроссовки adizero Evo SL M


In [ ]:
# обучаем голову

model = LitClassifier(
    model_name="resnet18",
    num_classes=df["sneaker_class"].nunique(),
    lr=1e-3,
    freeze_backbone=True
)

trainer = pl.Trainer(
    max_epochs=5,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name    ┃ Type             ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model   │ ResNet           │ 11.2 M │ train │     0 │
│ 1 │ loss_fn │ CrossEntropyLoss │      0 │ train │     0 │
└───┴─────────┴──────────────────┴────────┴───────┴───────┘

Trainable params: 21.0 K                                                                                           
Non-trainable params: 11.2 M                                                                                       
Total params: 11.2 M                                                                                               
Total estimated model params size (MB): 44                                                                         
Modules in train mode: 95                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)`
is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.

In [16]:
pred_batches = trainer.predict(model, dataloaders=[test_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in test_dataset]
print(classification_report(y_true, y_pred))


/opt/homebrew/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.00      0.00      0.00         1
           1       0.00      0.00      0.00         2
           2       0.00      0.00      0.00         5
           3       0.00      0.00      0.00         6
           4       0.00      0.00      0.00         2
           5       0.00      0.00      0.00         4
           6       0.00      0.00      0.00         1
           7       0.00      0.00      0.00         6
           8       0.00      0.00      0.00         6
           9       0.00      0.00      0.00         2
          10       0.00      0.00      0.00        10
          11       0.00      0.00      0.00         6
          12       0.00      0.00      0.00         4
          13       0.00      0.00      0.00        12
          14       0.00      0.00      0.00         8
          15       0.17      0.06      0.09        16
          16       0.50      0.25      0.33         4
          17       0.00    

/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-pa

In [17]:
pred_batches = trainer.predict(model, dataloaders=[val_loader])
y_pred = torch.cat(pred_batches).cpu().numpy()
y_true = [x[1].numpy().item() for x in val_dataset]
print(classification_report(y_true, y_pred))


/opt/homebrew/lib/python3.11/site-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

              precision    recall  f1-score   support

           0       0.40      0.43      0.41        54
           1       0.67      0.33      0.44        54
           2       0.63      0.63      0.63        54
           3       0.35      0.61      0.44        56
           4       0.47      0.62      0.54        56
           5       0.41      0.56      0.47        54
           6       0.48      0.55      0.52        58
           7       0.44      0.70      0.54        57
           8       0.64      0.63      0.64        54
           9       0.54      0.71      0.62        56
          10       0.40      0.53      0.46        53
          11       0.62      0.53      0.57        53
          12       0.53      0.86      0.65        56
          13       0.45      0.25      0.33        55
          14       0.48      0.71      0.57        52
          15       0.55      0.46      0.50        50
          16       0.62      0.29      0.40        55
          17       0.47    

/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/opt/homebrew/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [ ]:
# уменьшаем lr и дообучаем все

model.unfreeze()
model.lr = 1e-4

trainer = pl.Trainer(
    max_epochs=10,
    callbacks=callbacks
)

trainer.fit(model, train_loader, val_loader)